In [1]:
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# ── Model ────────────────────────────────────────────────────────────────────

class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, dropout_rate=0.0):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        return out

# ── Data ─────────────────────────────────────────────────────────────────────

def get_loaders(batch_size=64):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    train_ds = datasets.MNIST("data", train=True,  download=True, transform=transform)
    val_ds   = datasets.MNIST("data", train=False, download=True, transform=transform)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size)
    return train_loader, val_loader

# ── Training loop ─────────────────────────────────────────────────────────────

def train_and_evaluate(trial, params):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_loader, val_loader = get_loaders(batch_size=params["batch_size"])

    model = SimpleNN(
        input_size=784,
        hidden_size=params["hidden_size"],
        num_classes=10,
        dropout_rate=params["dropout_rate"],
    ).to(device)

    optimizer_cls = {"adam": optim.Adam, "sgd": optim.SGD, "rmsprop": optim.RMSprop}[params["optimizer"]]
    optimizer = optimizer_cls(model.parameters(), lr=params["lr"])
    criterion = nn.CrossEntropyLoss()

    for epoch in range(params["epochs"]):
        # ── train ──
        model.train()
        for images, labels in train_loader:
            images = images.view(-1, 784).to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()

        # ── validate ──
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.view(-1, 784).to(device)
                labels = labels.to(device)
                preds = model(images).argmax(dim=1)
                correct += (preds == labels).sum().item()
                total   += labels.size(0)

        val_accuracy = correct / total

        # Report intermediate value so the dashboard shows learning curves
        # and pruners can cut unpromising trials early
        trial.report(val_accuracy, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return val_accuracy

# ── Objective ─────────────────────────────────────────────────────────────────

def objective(trial: optuna.Trial) -> float:
    params = {
        # architecture
        "hidden_size":   trial.suggest_categorical("hidden_size", [128, 256, 512, 1024]),
        "dropout_rate":  trial.suggest_float("dropout_rate", 0.0, 0.5),
        # optimisation
        "optimizer":     trial.suggest_categorical("optimizer", ["adam", "sgd", "rmsprop"]),
        "lr":            trial.suggest_float("lr", 1e-4, 1e-1, log=True),
        "batch_size":    trial.suggest_categorical("batch_size", [32, 64, 128]),
        "epochs":        3,   # keep short for search; increase for final run
    }
    return train_and_evaluate(trial, params)

# ── Study ─────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # RDB storage lets optuna-dashboard read results in real time
    storage = "sqlite:///optuna_fnn.db"

    study = optuna.create_study(
        study_name="fnn-mnist",
        direction="maximize",          # we're maximising val accuracy
        storage=storage,
        load_if_exists=True,           # safe to re-run / resume
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
        sampler=optuna.samplers.TPESampler(seed=42),
    )

    study.optimize(objective, n_trials=30, timeout=600)

    # ── Results ───────────────────────────────────────────────────────────────
    best = study.best_trial
    print(f"\nBest trial #{best.number}")
    print(f"  Val accuracy : {best.value:.4f}")
    print(f"  Params       : {best.params}")

[I 2026-02-20 18:31:12,020] Using an existing study with name 'fnn-mnist' instead of creating a new one.
[I 2026-02-20 18:34:52,983] Trial 14 finished with value: 0.977 and parameters: {'hidden_size': 1024, 'dropout_rate': 0.39194800826273973, 'optimizer': 'rmsprop', 'lr': 0.0008643690720924116, 'batch_size': 32}. Best is trial 6 with value: 0.98.
[I 2026-02-20 18:38:32,316] Trial 15 finished with value: 0.9731 and parameters: {'hidden_size': 512, 'dropout_rate': 0.48876765116085846, 'optimizer': 'rmsprop', 'lr': 0.0007800640523253011, 'batch_size': 32}. Best is trial 6 with value: 0.98.
[I 2026-02-20 18:42:01,622] Trial 16 finished with value: 0.9757 and parameters: {'hidden_size': 512, 'dropout_rate': 0.47290485998685644, 'optimizer': 'rmsprop', 'lr': 0.0004243167233394771, 'batch_size': 32}. Best is trial 6 with value: 0.98.



Best trial #6
  Val accuracy : 0.9800
  Params       : {'hidden_size': 512, 'dropout_rate': 0.06101911742238941, 'optimizer': 'rmsprop', 'lr': 0.0005975027999960298, 'batch_size': 32}


In [3]:
import subprocess

dashboard_process = subprocess.Popen(
    ["optuna-dashboard", "sqlite:///optuna_fnn.db"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print("Dashboard running at http://127.0.0.1:8080")

Dashboard running at http://127.0.0.1:8080


In [ ]:
import optuna
import wandb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# ── Model ────────────────────────────────────────────────────────────────────

class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, dropout_rate=0.0):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        return out

# ── Data ─────────────────────────────────────────────────────────────────────

def get_loaders(batch_size=64):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    train_ds = datasets.MNIST("data", train=True,  download=True, transform=transform)
    val_ds   = datasets.MNIST("data", train=False, download=True, transform=transform)
    return DataLoader(train_ds, batch_size=batch_size, shuffle=True), \
           DataLoader(val_ds,   batch_size=batch_size)

# ── Training loop ─────────────────────────────────────────────────────────────

def train_and_evaluate(trial, params):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Each trial gets its own wandb run
    run = wandb.init(
        project="fnn-mnist-optuna",
        name=f"trial-{trial.number}",
        config=params,          # logs all hyperparams automatically
        reinit=True,            # needed so each trial starts a fresh run
    )

    train_loader, val_loader = get_loaders(batch_size=params["batch_size"])

    model = SimpleNN(
        input_size=784,
        hidden_size=params["hidden_size"],
        num_classes=10,
        dropout_rate=params["dropout_rate"],
    ).to(device)

    # Optional: watch gradients and model topology
    wandb.watch(model, log="all", log_freq=100)

    optimizer_cls = {"adam": optim.Adam, "sgd": optim.SGD, "rmsprop": optim.RMSprop}[params["optimizer"]]
    optimizer = optimizer_cls(model.parameters(), lr=params["lr"])
    criterion = nn.CrossEntropyLoss()

    for epoch in range(params["epochs"]):
        # ── train ──
        model.train()
        train_loss = 0
        for images, labels in train_loader:
            images = images.view(-1, 784).to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)

        # ── validate ──
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.view(-1, 784).to(device)
                labels = labels.to(device)
                preds = model(images).argmax(dim=1)
                correct += (preds == labels).sum().item()
                total   += labels.size(0)

        val_accuracy = correct / total

        # Log metrics to wandb per epoch
        wandb.log({
            "epoch":        epoch,
            "train_loss":   avg_train_loss,
            "val_accuracy": val_accuracy,
        })

        # Optuna pruning
        trial.report(val_accuracy, epoch)
        if trial.should_prune():
            run.finish(exit_code=1)   # mark pruned runs in wandb
            raise optuna.exceptions.TrialPruned()

    # Log the final best metric as a summary (shows up in wandb table)
    wandb.summary["best_val_accuracy"] = val_accuracy
    run.finish()

    return val_accuracy

# ── Objective ─────────────────────────────────────────────────────────────────

def objective(trial: optuna.Trial) -> float:
    params = {
        "hidden_size":  trial.suggest_categorical("hidden_size", [128, 256, 512, 1024]),
        "dropout_rate": trial.suggest_float("dropout_rate", 0.0, 0.5),
        "optimizer":    trial.suggest_categorical("optimizer", ["adam", "sgd", "rmsprop"]),
        "lr":           trial.suggest_float("lr", 1e-4, 1e-1, log=True),
        "batch_size":   trial.suggest_categorical("batch_size", [32, 64, 128]),
        "epochs":       3,
    }
    return train_and_evaluate(trial, params)

# ── Study ─────────────────────────────────────────────────────────────────────

# Cell 1 — create study
study = optuna.create_study(
    study_name="fnn-mnist",
    direction="maximize",
    storage="sqlite:///optuna_fnn.db",
    load_if_exists=True,
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
    sampler=optuna.samplers.TPESampler(seed=42),
)

# Cell 2 — launch optuna dashboard (optional, alongside wandb)
import subprocess
dashboard_process = subprocess.Popen(
    ["optuna-dashboard", "sqlite:///optuna_fnn.db"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

# Cell 3 — run (each trial logs to its own wandb run)
study.optimize(objective, n_trials=30)

# ── Print best result ─────────────────────────────────────────────────────────
best = study.best_trial
print(f"Best trial #{best.number}  →  val_accuracy: {best.value:.4f}")
print(f"Params: {best.params}")

[I 2026-02-20 19:10:51,663] Using an existing study with name 'fnn-mnist' instead of creating a new one.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Trish\_netrc.
wandb: Currently logged in as: trish-nair (trish-nair-carnegie-mellon-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


epoch,▁▅█
train_loss,█▂▁
val_accuracy,▁█▇
epoch,2
train_loss,0.11099
val_accuracy,0.9731


[I 2026-02-20 19:13:49,567] Trial 24 pruned. 


epoch,▁█
train_loss,█▁
val_accuracy,▁█
epoch,1
train_loss,0.09288
val_accuracy,0.9714


[I 2026-02-20 19:15:48,953] Trial 26 pruned. 


epoch,▁█
train_loss,█▁
val_accuracy,▁█
epoch,1
train_loss,0.183
val_accuracy,0.9667


[I 2026-02-20 19:17:30,874] Trial 28 pruned. 
